In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import numpy as np
import random
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
from PIL import Image
import pandas as pd

# ----------------------
# 1) Setup and Reproducibility
# ----------------------
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

# Define folder to save checkpoints and Grad-CAM outputs
date_str = datetime.now().strftime('%Y%m%d')
checkpoint_folder = f'/content/drive/My Drive/FYP/VGGModel/HQ3_{date_str}/'
os.makedirs(checkpoint_folder, exist_ok=True)

# ----------------------
# 2) Data Transforms & Datasets
# ----------------------
# Training augmentation (same as Code 1)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),  # VGG expects 224x224 input size
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomResizedCrop(300, scale=(0.95, 1.0)),
    transforms.RandomAdjustSharpness(sharpness_factor=1.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# For validation/test and Grad-CAM we use a fixed transform (no heavy augmentation)
transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Paths for training and testing datasets
data_path = '/content/drive/My Drive/FYP/Dataset/HQ3/train'
test_data_path = '/content/drive/My Drive/FYP/Dataset/HQ3/test'

# Use the full training dataset to later compute class weights and get sample images
full_train_dataset = datasets.ImageFolder(data_path, transform=transform_train)

# ----------------------
# 3) Compute Class Weights for Imbalanced Data
# ----------------------
targets = [s[1] for s in full_train_dataset.samples]
class_counts = Counter(targets)
total = sum(class_counts.values())
# Assuming two classes; adjust if necessary.
class_weights = torch.tensor([
    total / (2 * class_counts[0]),
    total / (2 * class_counts[1])
]).float().to(device)
print("Class weights:", class_weights)

# ----------------------
# 4) Train/Validation Split
# ----------------------
train_size = int(0.95 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Create data loaders (using training augmentation for train_loader;
# validation loader uses the fixed transform defined inside the dataset)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Test dataset uses the fixed transform
test_dataset = datasets.ImageFolder(test_data_path, transform=transform_val_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Get class names from the original dataset
class_names = full_train_dataset.classes

# ----------------------
# 5) Define the Modified VGG16 Model (same architecture as Code 1)
# ----------------------
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        from torchvision.models import VGG16_Weights
        self.vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        # Replace the final classifier layer with Dropout + Linear for binary classification
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_ftrs, 2)
        )
    def forward(self, x):
        return self.vgg(x)

model = VGG16Modified().to(device)

# Use weighted cross–entropy loss
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

# ----------------------
# 6) Grad-CAM & Utility Functions
# ----------------------
def generate_gradcam(model, image_tensor, target_layer):
    """
    Generates Grad-CAM for a given image and model.
    Returns:
      cam (normalized 2D numpy array),
      predicted_class_idx (int),
      predicted_confidence (float)
    """
    model.eval()
    features = []
    grads = []

    def forward_hook(module, input, output):
        features.append(output)

    def backward_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    # Register hooks on the target layer
    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_backward_hook(backward_hook)

    # Forward pass
    outputs = model(image_tensor)
    probs = F.softmax(outputs, dim=1)
    predicted_class_idx = torch.argmax(probs, dim=1).item()
    predicted_confidence = probs[0, predicted_class_idx].item()

    # Backward pass for the predicted class
    model.zero_grad()
    class_score = outputs[0, predicted_class_idx]
    class_score.backward()

    # Extract gradients and feature maps
    gradient = grads[0].detach().cpu().numpy()[0]  # shape: (C, H, W)
    feature_map = features[0].detach().cpu().numpy()[0]  # shape: (C, H, W)

    # Remove hooks
    forward_handle.remove()
    backward_handle.remove()

    # Compute weights via global-average pooling
    weights = np.mean(gradient, axis=(1, 2))  # shape: (C,)
    cam = np.zeros(feature_map.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * feature_map[i, :, :]

    # Apply ReLU and normalize the CAM
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (image_tensor.shape[3], image_tensor.shape[2]))
    cam -= np.min(cam)
    cam /= (cam.max() + 1e-9)
    return cam, predicted_class_idx, predicted_confidence

def preprocess_image(image_path):
    """
    Loads an image from a given path and applies the fixed transform.
    Returns a tensor of shape (1, 3, 224, 224) on the specified device.
    """
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform_val_test(image).unsqueeze(0).to(device)
    return image_tensor

def save_visualization(image_path, cam, predicted_class, confidence, true_class, save_path):
    """
    Creates an overlay of the Grad-CAM heatmap on the original image,
    adds text annotations (true class, predicted class, correctness, confidence),
    and saves the result to 'save_path'.
    """
    # Load and resize original image
    original_img = Image.open(image_path).resize((224, 224), Image.LANCZOS)
    original_np = np.array(original_img)

    # Create heatmap from CAM
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Blend heatmap with the original image
    overlay = np.clip(0.5 * original_np + 0.5 * heatmap, 0, 255).astype(np.uint8)

    # Prepare annotation text
    pred_label = class_names[predicted_class]
    correct = (predicted_class == true_class)
    confidence_text = f"{confidence*100:.2f}%"
    result_text = f"True: {class_names[true_class]} | Pred: {pred_label} | Conf: {confidence_text} | {'Correct' if correct else 'Wrong'}"

    # Put text on the image using cv2.putText
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.putText(
        overlay_bgr,
        result_text,
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0) if correct else (0, 0, 255),
        2,
        cv2.LINE_AA
    )
    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    cv2.imwrite(save_path, cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))

def get_sample_images(dataset, num_samples_per_class=3):
    """
    Returns a dict mapping each class index to a list of image paths.
    """
    sample_dict = {}
    for path, label in dataset.samples:
        if label not in sample_dict:
            sample_dict[label] = []
        if len(sample_dict[label]) < num_samples_per_class:
            sample_dict[label].append(path)
    return sample_dict

def run_gradcam_samples(model, sample_images, epoch, target_layer):
    """
    For each class and each sample image, run Grad-CAM and save the visualization.
    """
    gradcam_epoch_folder = os.path.join(checkpoint_folder, f"gradcam_epoch_{epoch+1}")
    os.makedirs(gradcam_epoch_folder, exist_ok=True)

    for cls_idx, paths in sample_images.items():
        for i, image_path in enumerate(paths):
            image_tensor = preprocess_image(image_path)
            cam, pred_class, confidence = generate_gradcam(model, image_tensor, target_layer)
            save_path = os.path.join(gradcam_epoch_folder, f"class{cls_idx}_sample{i+1}.png")
            save_visualization(image_path, cam, pred_class, confidence, cls_idx, save_path)
            print(f"Saved Grad-CAM for class {cls_idx} sample {i+1} at epoch {epoch+1}")

# Prepare sample images (using the full training set for consistency)
sample_images = get_sample_images(full_train_dataset, num_samples_per_class=3)

# For Grad-CAM, choose the last convolutional layer of VGG16.
# (VGG16.features[28] is a convolutional layer before the final ReLU/MaxPool.)
target_layer = model.vgg.features[28]

# ----------------------
# 7) Training, Validation, and Testing Functions
# ----------------------
def validate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100.0 * correct / total
    print(f'Validation Accuracy: {accuracy:.2f}%')
    return accuracy

def train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=30):
    train_losses = []
    val_accuracies = []
    checkpoint_model_path = os.path.join(checkpoint_folder, f"checkpoint_model_vgg_{date_str}.pth")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)
        print(f"Epoch [{epoch+1}/{epochs}] - Training Loss: {train_loss:.6f}")

        # Validate the model
        val_acc = validate(model, val_loader)
        val_accuracies.append(val_acc)
        scheduler.step(val_acc)

        # Save checkpoint (overwrite each epoch)
        torch.save(model.state_dict(), checkpoint_model_path)
        print(f"Checkpoint saved for epoch {epoch+1} at {checkpoint_model_path}")

        # Run Grad-CAM on sample images and save visualizations
        run_gradcam_samples(model, sample_images, epoch, target_layer)

    # Plot training & validation metrics
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('Validation Accuracy')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    # Save final model
    final_model_path = os.path.join(checkpoint_folder, f"final_model_vgg_{date_str}.pth")
    torch.save(model.state_dict(), final_model_path)
    print(f"Final model saved at {final_model_path}")

def test(model, loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100.0 * correct / total
    print(f'Test Loss: {test_loss/len(loader):.6f}, Test Accuracy: {accuracy:.2f}%')

# ----------------------
# 8) Main Routine: Train and Test
# ----------------------
if __name__ == "__main__":
    train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=35)
    test(model, test_loader, criterion)


Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class weights: tensor([3.1387, 0.5947], device='cuda:0')


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:06<00:00, 85.1MB/s]


Epoch [1/35] - Training Loss: 0.640805
Validation Accuracy: 94.74%
Checkpoint saved for epoch 1 at /content/drive/My Drive/FYP/VGGModel/HQ3_20250218/checkpoint_model_vgg_20250218.pth


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1827: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Saved Grad-CAM for class 0 sample 1 at epoch 1
Saved Grad-CAM for class 0 sample 2 at epoch 1
Saved Grad-CAM for class 0 sample 3 at epoch 1
Saved Grad-CAM for class 1 sample 1 at epoch 1
Saved Grad-CAM for class 1 sample 2 at epoch 1
Saved Grad-CAM for class 1 sample 3 at epoch 1
Epoch [2/35] - Training Loss: 0.760435
Validation Accuracy: 92.11%
Checkpoint saved for epoch 2 at /content/drive/My Drive/FYP/VGGModel/HQ3_20250218/checkpoint_model_vgg_20250218.pth
Saved Grad-CAM for class 0 sample 1 at epoch 2
Saved Grad-CAM for class 0 sample 2 at epoch 2
Saved Grad-CAM for class 0 sample 3 at epoch 2
Saved Grad-CAM for class 1 sample 1 at epoch 2
Saved Grad-CAM for class 1 sample 2 at epoch 2
Saved Grad-CAM for class 1 sample 3 at epoch 2
Epoch [3/35] - Training Loss: 0.386083
Validation Accuracy: 89.47%
Checkpoint saved for epoch 3 at /content/drive/My Drive/FYP/VGGModel/HQ3_20250218/checkpoint_model_vgg_20250218.pth
Saved Grad-CAM for class 0 sample 1 at epoch 3
Saved Grad-CAM for clas